In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import AGEMTrainer, IntervalAGEMTrainer
from src.data_utils import get_mnist_tasks, _extract_targets, get_context_sets
from src.utils.general import InContextHead
from src import models
from src.regulariser import UnbiasRegulariser, L2Regulariser, MultiRegulariser

from configs import MNIST_AGEM_CONFIG as CONFIG

### Task Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda", seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)


In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=100,
    paradigm="TIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
        context_id=i,
    )
    agem_trainer.test(test_tasks, context_list=list(range(len(test_tasks))))

### Domain Incremental Learning

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=2, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=100,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)

### Class Incremental Training

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
agem_trainer = AGEMTrainer(
    model,
    memory_samples=200,
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)

## IntervalAGEMTrainer

### Class Incremental Learning

In [ ]:
from configs import MNIST_IT_CONFIG as INTERVAL_CONFIG

In [ ]:
SEED = 0
train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

model = models.get_mnist_model(device="cuda", output_dim=10, seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
agem_trainer = IntervalAGEMTrainer(
    model,
    memory_samples=200,
    checkpoint=INTERVAL_CONFIG["checkpoint"],
    n_iters=INTERVAL_CONFIG["n_iters"],
    min_acc_limit=INTERVAL_CONFIG["min_acc_limit"],
    min_acc_increment=INTERVAL_CONFIG["min_acc_increment"],
    primal_learning_rate=INTERVAL_CONFIG["primal_learning_rate"],
    dual_learning_rate=INTERVAL_CONFIG["dual_learning_rate"],
    projection_strategy=INTERVAL_CONFIG["projection_strategy"],
    n_certificate_samples=INTERVAL_CONFIG["n_certificate_samples"],
    penalty_coefficient=INTERVAL_CONFIG["penalty_coefficient"],
    paradigm="CIL",
    seed=SEED,
)

for i, (train, val, test) in enumerate(zip(train_tasks, val_tasks, test_tasks)):
    agem_trainer.train(
        train,
        val,
        batch_size=CONFIG["batch_size"],
        epochs=CONFIG["epochs"],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    agem_trainer.test(test_tasks)
    if i < len(test_tasks) - 1:
        agem_trainer.compute_rashomon_set(test)

# Ablation Study

In [6]:
import wandb

In [ ]:
# HYPERPARAMS
""" TIL
MNIST = {
    "batch_size": 256,
    "epochs": 3,
    "lr": 0.001,
    "weight_decay": 0,
    "unbias_lambda": 0.01,
    "l2_lambda": 0.01
}
"""

for paradigm in ["TIL", "DIL", "CIL"]:
    for i in range(5):
        failed = False
        with wandb.init(
            project="certified-continual-learning",
            reinit=True,
            tags=["final_mnist_buffer", "buffer_agem", f"buffer_{paradigm.lower()}"],
        ):
            def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
                """Map the global label to the in context label."""
                return labels % 2
            wandb.log({"seed": i})
            SEED = i
            train_tasks, _, test_tasks = get_mnist_tasks(seed=SEED, train_val_split_ratio=0.3, emnist=True)
            context_sets = get_context_sets(test_tasks)
            head = InContextHead(context_sets, 10, device="cuda")
            head.set_context(0)
            model = models.get_mnist_model(device="cuda", output_dim=2 if paradigm == "DIL" else 10, seed=SEED, head = head if paradigm=="TIL" else None)
            print(
                f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
            )

            agem_trainer = AGEMTrainer(
                model,
                memory_samples=3750,
                paradigm=paradigm,
                seed=SEED,
                domain_map_fn=domain_map_fn if paradigm == "DIL" else None
            )

            acc_matrix = []
            for i, (train, test) in enumerate(
                zip(train_tasks, test_tasks)
            ):
                agem_trainer.train(
                    train,
                    test,
                    batch_size=CONFIG["batch_size"],
                    epochs=CONFIG["epochs"],
                    lr=CONFIG["lr"],
                    weight_decay=CONFIG["weight_decay"],
                    context_id=i if paradigm == "TIL" else None,
                    val_freq=int(len(train) / CONFIG["batch_size"]) - 1
                )
                results = agem_trainer.test(
                    test_tasks,
                    context_list=list(range(len(test_tasks)))
                    if paradigm == "TIL"
                    else [None] * len(test_tasks),
                )
                accs = [res[1] for res in results]
                if not i and accs[0] < 0.7:
                    wandb.finish(1)
                    failed = True
                    break
                acc_matrix.append(accs)

            if not failed:
                columns = [f"Test Task {i}" for i in range(len(test_tasks))]
                rows = [f"Task {i}" for i in range(len(test_tasks))]
                wandb.log(
                    {
                        "accuracy_matrix": wandb.Table(
                            data=acc_matrix, columns=columns, rows=rows
                        )
                    }
                )
                wandb.finish()

Tasks: [[7, 8], [0, 9], [1, 2], [3, 6], [4, 5]]


Training Epochs: 100%|███████████████████████████████████████████████████████████████████████████████| 3/3 [00:06<00:00,  2.22s/it, loss=0.0370, acc=0.9844]


Test Results: [(0.0169, 0.9935), (0.4225, 0.8306), (2.1401, 0.5069), (4.8331, 0.5235), (5.8074, 0.4497)] (Avg: (2.6440, 0.6608))


Training Epochs: 100%|███████████████████████████████████████████████████████████████████████████████| 3/3 [00:17<00:00,  5.77s/it, loss=0.0626, acc=0.9958]


Test Results: [(0.0257, 0.9911), (0.0421, 0.9881), (1.2856, 0.5649), (2.7811, 0.5769), (5.3607, 0.1547)] (Avg: (1.8990, 0.6551))


...

Training Epochs: 100%|███████████████████████████████████████████████████████████████████████████████| 3/3 [00:39<00:00, 13.07s/it, loss=0.2804, acc=0.9531]


Test Results: [(1.0913, 0.5563), (1.4987, 0.401), (1.4344, 0.4554), (0.251, 0.9399), (35.16, 0.0)] (Avg: (7.8871, 0.4705))


Training Epochs: 100%|███████████████████████████████████████████████████████████████████████████████| 3/3 [00:50<00:00, 16.86s/it, loss=3.1069, acc=0.0000]


Test Results: [(1.0292, 0.8729), (1.2709, 0.8049), (1.3475, 0.8069), (1.4764, 0.7376), (3.0319, 0.0)] (Avg: (1.6312, 0.6445))


seed,▁
seed,4
